In [1]:
# Check with people which is the correct pedestal file to use
# make sure that the unique id -> pixel id is taking into account the tile swaps
# why does the pedestal file have channels that are supposed to be nonrouted?
# it looks like the pedestal file I'm using is using tile number instead of io_channel

In [1]:
import json
import numpy as np
import tqdm
import matplotlib.pyplot as plt

In [3]:
with open('/global/common/software/dune/mkramer/devel/flow4pedestal/reference-cold-pedestal-2024_06_05_08_28_19_CDTevd_ped.tile_id.decimal.json', 'r') as f:
    data = json.load(f)
    

In [4]:
def unique_channel_id(d):
    return ((d['io_group'].astype(int)*10000+((d['io_channel'].astype(int)-1)//4)+1)*1000 \
            + d['chip_id'].astype(int))*100 + d['channel_id'].astype(int)

def unique_to_channel_id(unique):
    return unique % 100

def unique_to_chip_id(unique):
    return (unique// 100) % 1000

def unique_to_io_channel(unique):
    return(unique//(100*1000)) % 1000

def unique_to_tiles(unique):
    return ( (unique_to_io_channel(unique)-1) // 4) + 1

def unique_to_io_group(unique):
    return(unique // (100*1000*10000)) % 10000

In [5]:
def convert_unique_id(unique_id):
    io_group = (unique_to_io_group(unique_id) - 1) % 2 + 1
    tile_id = (unique_to_io_channel(unique_id) - 1) % 8 + 1
    chip_id = unique_to_chip_id(unique_id)
    channel_id = unique_to_channel_id(unique_id)

    convert = (
        (io_group*10000+tile_id)*1000 \
            + chip_id)*100 + channel_id

    return convert

In [6]:
with open('uniqueid_to_pixelid_mod0.json', 'r') as file:
    unique_id_to_pixel_id_mod0 = json.load(file)
with open('uniqueid_to_pixelid_mod1.json', 'r') as file:
    unique_id_to_pixel_id_mod1 = json.load(file)
with open('uniqueid_to_pixelid_mod2.json', 'r') as file:
    unique_id_to_pixel_id_mod2 = json.load(file)
with open('uniqueid_to_pixelid_mod3.json', 'r') as file:
    unique_id_to_pixel_id_mod3 = json.load(file)

In [7]:
i = 0
pixel_ids = [[],[],[],[]] 
pedestals = [[],[],[],[]]
nonrouted_v2a_channels = [6, 7, 8, 9, 22, 23, 24, 25, 38, 39, 40, 54, 55, 56, 57]


for k, v in data.items():
    
    
    # unique_id = int(k)
    pedestal = v['pedestal_mv']

    unique_id = int(k)

    unique_iog = unique_to_io_group(unique_id)
    chip_id = unique_to_chip_id(unique_id)
    converted_unique_id = convert_unique_id(unique_id)
    channel_id = unique_to_channel_id(unique_id)
    # tile_id = unique_to_io_channel(unique_id)
    

    if chip_id not in range(11,111):
        continue
        
    if (unique_iog in [1,2]) and (channel_id not in nonrouted_v2a_channels):
        mod = 0
        buff = 0
        pixel_id = buff + unique_id_to_pixel_id_mod0[str(converted_unique_id)]
        
    elif (unique_iog in [3,4]) and (channel_id not in nonrouted_v2a_channels) :
        mod = 1
        buff = 7*10*2 * 7*10*4 * 2
        pixel_id = buff + unique_id_to_pixel_id_mod1[str(converted_unique_id)]
        
    elif (unique_iog in [5,6]) :
        mod = 2
        buff = 8*10*2 * 8*10*4 * 2 * 2
        pixel_id = buff + unique_id_to_pixel_id_mod2[str(converted_unique_id)]
        
    elif (unique_iog in [7,8]) and (channel_id not in nonrouted_v2a_channels):
        mod = 3
        buff = 7*10*2 * 7*10*4 * 2 * 3
        pixel_id = buff + unique_id_to_pixel_id_mod3[str(converted_unique_id)]
        
    else:
        continue
    try:
        pixel_ids[mod].append(pixel_id)
        pedestals[mod].append(pedestal)
    except:
        break
        
    i+=1

for i in range(4):
    print(i)
    out_file = 'pedestals_module{i}.npz'.format(i=i)
    keys = np.array(pixel_ids[i], dtype='int64')
    values = np.array(pedestals[i], dtype='float64')
    default = np.array([580.])
    np.savez(out_file, keys=keys, values=values, default=default)
# print(pixel_ids)

0
1
2
3


In [8]:
print( np.array(pixel_ids[0]).min(), np.array(pixel_ids[0]).max())
print( np.array(pixel_ids[1]).min(), np.array(pixel_ids[1]).max())
print( np.array(pixel_ids[2]).min(), np.array(pixel_ids[2]).max())
print( np.array(pixel_ids[3]).min(), np.array(pixel_ids[3]).max())

0 78399
78400 156799
204800 307199
235200 313599
